## STC 任务原始数据处理

In [1]:
import os
import sys
import numpy as np
import csv
import pandas as pd
import matplotlib.pyplot as plt
import cv2 as cv
from tqdm import tqdm
from loguru import logger
from datetime import datetime, timedelta, timezone
from skyfield.api import load

logger.remove()
logger.add(sys.stderr, format="{time:YYYY-MM-DD HH:mm:ss} {level} {message}", level="INFO")

1

In [2]:
#! 设置全局变量
RAW_DATA_PATH = os.path.join('/home/nvidia/ssd/Projects/CrossView_ws/data/STC')
INSTA360_PATH = os.path.join(RAW_DATA_PATH, 'Insta360')
INSTA360_VIDEOS = ['VID_20230419_173014/pano.mp4', 'VID_20230419_175200/pano.mp4']  # 相对于 Insta360_PATH 的路径
INSTA360_START_TIMES = [1681896617381945, 1681897923264599]
INSTA360_FPS = 29.97
LADYBUG_PATH = os.path.join(RAW_DATA_PATH, 'Ladybug')
LADYBUG_VIDEOS = ['Ladybug-stream_Panoramic_3500x1750_9000kbps.mp4']  # 相对于 LADYBUG_PATH 的路径
LADYBUG_START_TIMES = ['2023-04-19-09:51:43.195']
LADYBUG_FPS = 10

In [3]:
# 视频拆分为帧序列并生成时间戳文件
def process_videos(video_file, csv_file, fps, start_time):
    # 计算总帧数
    cap = cv.VideoCapture(video_file)
    if not cap.isOpened():
        logger.error(f'Fail to open video file: {video_file}')
        return 0
    else:
        logger.info(f'Open video file: {video_file}')
    total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
    logger.info(f'Total frames: {total_frames}')
    
    # 生成帧编号和时间戳数据
    timestamps = [start_time]
    time_per_frame = 1 / fps
    for i in range(1, total_frames):
        timestamps.append(timestamps[-1] + time_per_frame)
    data = {
        "Frame": range(total_frames),
        "Timestamp": timestamps
    }
    df = pd.DataFrame(data)
    df.to_csv(csv_file, index=False, header=None)

    # 拆分视频为帧序列
    save_dir = os.path.join(os.path.dirname(video_file), 'imgs')
    os.makedirs(save_dir, exist_ok=True)
    for idx in tqdm(range(total_frames)):
        ret, frame = cap.read()
        if not ret:
            break  # 视频结束或读取错误
        frame_filename = os.path.join(save_dir, f"{idx}.jpg")
        cv.imwrite(frame_filename, frame)

    cap.release()


### 1. 处理 Insta360 数据

In [4]:
# 验证相机时间戳
camera_timestamp = 1681897923264599 / 1e6  # 单位微秒，转换为秒
camera_date_utc = datetime.utcfromtimestamp(camera_timestamp)  # UTC 时间
print("UTC time: ", camera_date_utc)
beijing_timezone = timezone(timedelta(hours=8))
camera_date_beijing = camera_date_utc.replace(tzinfo=timezone.utc).astimezone(beijing_timezone)  # 北京时间
print("Beijing time: ", camera_date_beijing)

UTC time:  2023-04-19 09:52:03.264599
Beijing time:  2023-04-19 17:52:03.264599+08:00


In [5]:
# 处理图像数据
for i, video_file in enumerate(INSTA360_VIDEOS):
    video_file = os.path.join(INSTA360_PATH, video_file)
    print(video_file)
    start_time = INSTA360_START_TIMES[i] / 1e6  # 单位微秒，转换为秒
    csv_file = video_file.replace('.mp4', '.csv')
    process_videos(video_file, csv_file, INSTA360_FPS, start_time)

2024-01-16 19:08:04 INFO Open video file: /home/nvidia/ssd/Projects/CrossView_ws/data/STC/Insta360/VID_20230419_173014/pano.mp4
2024-01-16 19:08:04 INFO Total frames: 5160


/home/nvidia/ssd/Projects/CrossView_ws/data/STC/Insta360/VID_20230419_173014/pano.mp4


100%|██████████| 5160/5160 [10:27<00:00,  8.23it/s]
2024-01-16 19:18:31 INFO Open video file: /home/nvidia/ssd/Projects/CrossView_ws/data/STC/Insta360/VID_20230419_175200/pano.mp4
2024-01-16 19:18:31 INFO Total frames: 25562


/home/nvidia/ssd/Projects/CrossView_ws/data/STC/Insta360/VID_20230419_175200/pano.mp4


100%|██████████| 25562/25562 [51:59<00:00,  8.20it/s]  


### 2. 处理 LadyBug 数据

In [8]:
# 验证相机时间戳
camera_date_str = '2023-04-19-09:51:43.195'
# 使用 strptime 将字符串转换为 datetime 对象，并设置为UTC时区
datetime_obj = datetime.strptime(camera_date_str, "%Y-%m-%d-%H:%M:%S.%f")
datetime_obj = datetime_obj.replace(tzinfo=timezone.utc)
# 获取UTC时间的时间戳
timestamp_utc = datetime_obj.timestamp()
camera_date_utc = datetime.utcfromtimestamp(timestamp_utc)  # UTC 时间
print("UTC time: ", camera_date_utc)
beijing_timezone = timezone(timedelta(hours=8))
camera_date_beijing = camera_date_utc.replace(tzinfo=timezone.utc).astimezone(beijing_timezone)  # 北京时间
print("Beijing time: ", camera_date_beijing)

UTC time:  2023-04-19 09:51:43.195000
Beijing time:  2023-04-19 17:51:43.195000+08:00


In [9]:
# 处理图像数据
for i, video_file in enumerate(LADYBUG_VIDEOS):
    video_file = os.path.join(LADYBUG_PATH, video_file)
    print(video_file)
    start_time = datetime.strptime(LADYBUG_START_TIMES[i], "%Y-%m-%d-%H:%M:%S.%f").replace(tzinfo=timezone.utc).timestamp()  # 单位秒
    csv_file = video_file.replace('.mp4', '.csv')
    process_videos(video_file, csv_file, LADYBUG_FPS, start_time)

2024-01-16 20:19:18 INFO Open video file: /home/nvidia/ssd/Projects/CrossView_ws/data/STC/Ladybug/Ladybug-stream_Panoramic_3500x1750_9000kbps.mp4
2024-01-16 20:19:18 INFO Total frames: 4386


/home/nvidia/ssd/Projects/CrossView_ws/data/STC/Ladybug/Ladybug-stream_Panoramic_3500x1750_9000kbps.mp4


100%|██████████| 4386/4386 [07:15<00:00, 10.07it/s]
